# Поиск дублей объявлений — Avito ML Cup 2025


In [ ]:
import numpy as np

__all__ = ["mean_average_precision", "average_precision", "map_by_group"]


def average_precision(relevance: np.ndarray) -> float:
    # relevance — 0/1 в порядке убывания скора
    rel = np.asarray(relevance, dtype=np.float64)
    n_pos = rel.sum()
    if n_pos == 0:
        return 0.0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / n_pos)


def map_by_group(
    groups: np.ndarray,
    y_true: np.ndarray,
    scores: np.ndarray,
    skip_empty: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # AP по всем группам за одну сортировку — цикл по группам на миллионах пар не живёт
    groups = np.asarray(groups)
    y = np.asarray(y_true).astype(np.float64)
    s = np.asarray(scores, dtype=np.float64)
    if not (len(groups) == len(y) == len(s)):
        raise ValueError("groups, y_true и scores должны быть одной длины")
    if len(groups) == 0:
        empty = np.array([])
        return empty, empty, empty

    # первичный ключ — группа, вторичный — скор по убыванию; в lexsort главный ключ последний
    order = np.lexsort((-s, groups))
    g = groups[order]
    y = y[order]

    n = len(y)
    starts = np.flatnonzero(np.concatenate(([True], g[1:] != g[:-1])))
    sizes = np.diff(np.concatenate((starts, [n])))
    row_start = np.repeat(starts, sizes)  # начало своей группы для каждой строки
    rank = np.arange(n) - row_start + 1  # позиция внутри группы, с 1

    # cum_excl[i] = сумма y[:i], значит попаданий до i включительно = cum_excl[i+1] - cum_excl[start]
    cum_excl = np.concatenate(([0.0], np.cumsum(y)))
    hits = cum_excl[1:] - cum_excl[row_start]

    contrib = hits / rank * y
    ap_sum = np.add.reduceat(contrib, starts)
    n_pos = np.add.reduceat(y, starts)
    ap = np.divide(ap_sum, n_pos, out=np.zeros_like(ap_sum), where=n_pos > 0)

    group_ids = g[starts]
    if skip_empty:
        # базы без единого дубля не в счёт, для них AP не определён
        keep = n_pos > 0
        return group_ids[keep], ap[keep], n_pos[keep]
    return group_ids, ap, n_pos


def mean_average_precision(
    groups: np.ndarray,
    y_true: np.ndarray,
    scores: np.ndarray,
    skip_empty: bool = True,
) -> float:
    # среднее AP по базам. на сплошных ties зависит от порядка строк, для константы бессмысленна
    _, ap, _ = map_by_group(groups, y_true, scores, skip_empty=skip_empty)
    return float(ap.mean()) if len(ap) else 0.0

## нормализация текста и починка гомоглифов

In [ ]:
"""тексты обфусцированы: часть кириллицы подменена латинскими двойниками.

fold_confusables сводит обе азбуки к одной — когда сравниваю строки между собой.
restore_homoglyphs чинит слово в сторону его алфавита — перед подачей в модель.
"""


import re
import unicodedata
from typing import Iterable, Sequence

__all__ = [
    "fold_confusables",
    "restore_homoglyphs",
    "normalize",
    "tokenize",
    "jaccard",
    "char_ngrams",
    "normalize_vocabulary",
    "CONFUSABLES",
    "FOLD_PAIRS",
]

# пары «латинская — кириллическая», неразличимые в типичном шрифте. заглавные и строчные
# отдельно: К и K неразличимы, а к и k — вполне
CONFUSABLES: tuple[tuple[str, str], ...] = (
    ("a", "а"), ("c", "с"), ("e", "е"), ("o", "о"), ("p", "р"), ("x", "х"), ("y", "у"),
    ("A", "А"), ("B", "В"), ("C", "С"), ("E", "Е"), ("H", "Н"), ("K", "К"), ("M", "М"),
    ("O", "О"), ("P", "Р"), ("T", "Т"), ("X", "Х"), ("Y", "У"),
)

FOLD_PAIRS: tuple[tuple[str, str], ...] = (
    ("а", "a"), ("с", "c"), ("е", "e"), ("о", "o"), ("р", "p"), ("х", "x"), ("у", "y"),
    ("к", "k"), ("м", "m"), ("н", "h"), ("в", "b"), ("т", "t"),
)

_LATIN_AMBIGUOUS = {lat for lat, _ in CONFUSABLES}
_CYRILLIC_AMBIGUOUS = {cyr for _, cyr in CONFUSABLES}

_TO_CYRILLIC = str.maketrans({lat: cyr for lat, cyr in CONFUSABLES})
_TO_LATIN = str.maketrans({cyr: lat for lat, cyr in CONFUSABLES})
_FOLD = str.maketrans({cyr: lat for cyr, lat in FOLD_PAIRS})

_LATIN_RE = re.compile(r"[A-Za-z]")
_CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")
_WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)
_PUNCT_RE = re.compile(r"[^\w\s]|_", re.UNICODE)
_SPACE_RE = re.compile(r"\s+")


def fold_confusables(text: str) -> str:
    # ждёт нижний регистр. свёртка разрушающая, но починенных совпадений на порядки больше
    if not text:
        return ""
    return text.translate(_FOLD)


def _restore_token(token: str) -> str:
    # голосуют только однозначные буквы, у которых двойника нет
    has_real_cyrillic = False
    has_real_latin = False
    for ch in token:
        if ch not in _CYRILLIC_AMBIGUOUS and _CYRILLIC_RE.match(ch):
            has_real_cyrillic = True
        elif ch not in _LATIN_AMBIGUOUS and _LATIN_RE.match(ch):
            has_real_latin = True
    if has_real_cyrillic and not has_real_latin:
        return token.translate(_TO_CYRILLIC)
    if has_real_latin and not has_real_cyrillic:
        return token.translate(_TO_LATIN)
    # либо осмысленная смесь (iPhone), либо сплошные двойники (ecco, сор) — угадывать нечего
    return token


def restore_homoglyphs(text: str) -> str:
    # куpткa зимняя ecco -> куртка зимняя ecco, бренд остаётся латинским
    if not text:
        return ""
    return _WORD_RE.sub(lambda m: _restore_token(m.group(0)), text)


def normalize(text: str, *, fold: bool = True) -> str:
    # nfkc, нижний регистр, пунктуация в пробелы, пробелы схлопнуты; свёртка последней
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    if fold:
        text = fold_confusables(text)
    text = _PUNCT_RE.sub(" ", text)
    return _SPACE_RE.sub(" ", text).strip()


def tokenize(text: str, *, fold: bool = True) -> list[str]:
    # числа оставляю намеренно: в объявлениях они несут модель, размер и объём памяти —
    # самое различающее в парах вроде iphone 11 256 гб против iphone 11 128 гб
    return normalize(text, fold=fold).split()


def jaccard(a: Iterable[str], b: Iterable[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    union = len(sa | sb)
    return len(sa & sb) / union if union else 0.0


def char_ngrams(text: str, n: int = 3) -> set[str]:
    # устойчивы к опечаткам и другому порядку слов, значит ловят переписанные описания
    # там, где пословный жаккар уже проваливается
    s = normalize(text)
    if len(s) < n:
        return {s} if s else set()
    return {s[i : i + n] for i in range(len(s) - n + 1)}


def normalize_vocabulary(texts: Sequence[str], func) -> dict[str, str]:
    # уникальных токенов в корпусе на два порядка меньше, чем вхождений. считать по
    # словарю, а не по каждому вхождению — это секунды против часов на миллионах строк
    vocab: dict[str, str] = {}
    for text in texts:
        if not text:
            continue
        for token in _WORD_RE.findall(text):
            if token not in vocab:
                vocab[token] = func(token)
    return vocab

## парные признаки

In [ ]:
"""признаки описывают пару, а не объявление: модель видит только сходство base и cand."""


import json
from typing import Iterable

import numpy as np
import pandas as pd


__all__ = [
    "build_features",
    "TEXT_PAIRS",
    "add_embedding_features",
    "embedding_pair_features",
    "side_numeric_features",
]

# поле и размер символьной n-граммы. у заголовков n=3, они короткие и длиннее просто
# не наберётся; у описаний n=4 — текста хватает, а случайных совпадений меньше
TEXT_PAIRS: tuple[tuple[str, int], ...] = (("title", 3), ("description", 4))

MAX_DESCRIPTION_CHARS = 512


def _safe_str(series: pd.Series) -> np.ndarray:
    return series.fillna("").astype(str).to_numpy()


def _containment(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


def _lcp_ratio(a: str, b: str) -> float:
    # общий префикс — ловит iphone 11 128 гб против iphone 11 256 гб
    if not a or not b:
        return 0.0
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i / max(len(a), len(b))


def _ngrams(text: str, n: int) -> set[str]:
    if len(text) < n:
        return {text} if text else set()
    return {text[i : i + n] for i in range(len(text) - n + 1)}


class _BoundedCache:
    """пары одной базы идут подряд, поэтому кэш разобранных текстов окупается.
    но только ограниченный: без потолка он разросся до 12.6 гб на одном файле.
    """

    __slots__ = ("_prepare", "_maxsize", "_data")

    def __init__(self, prepare, maxsize: int = 4096) -> None:
        self._prepare = prepare
        self._maxsize = maxsize
        self._data: dict = {}

    def get(self, raw: str):
        hit = self._data.get(raw)
        if hit is None:
            hit = self._prepare(raw)
            if len(self._data) >= self._maxsize:
                self._data.clear()
            self._data[raw] = hit
        return hit


def _text_similarity_block(
    base_raw: np.ndarray, cand_raw: np.ndarray, field: str, ngram: int, max_chars: int | None
) -> dict[str, np.ndarray]:
    n = len(base_raw)
    out = {
        f"{field}_exact": np.zeros(n, dtype=np.float32),
        f"{field}_jaccard_tok": np.zeros(n, dtype=np.float32),
        f"{field}_containment_tok": np.zeros(n, dtype=np.float32),
        f"{field}_jaccard_ngram": np.zeros(n, dtype=np.float32),
        f"{field}_digits_jaccard": np.zeros(n, dtype=np.float32),
        f"{field}_lcp_ratio": np.zeros(n, dtype=np.float32),
        f"{field}_len_ratio": np.zeros(n, dtype=np.float32),
        f"{field}_len_diff": np.zeros(n, dtype=np.float32),
        f"{field}_both_empty": np.zeros(n, dtype=np.float32),
    }
    def prepare(raw: str):
        norm = normalize(raw[:max_chars] if max_chars else raw)
        toks = norm.split()
        return norm, set(toks), _ngrams(norm, ngram), {t for t in toks if t.isdigit()}

    cache = _BoundedCache(prepare)

    for i in range(n):
        b_norm, b_tok, b_ng, b_dig = cache.get(base_raw[i])
        c_norm, c_tok, c_ng, c_dig = cache.get(cand_raw[i])

        if not b_norm and not c_norm:
            out[f"{field}_both_empty"][i] = 1.0
            continue

        out[f"{field}_exact"][i] = float(b_norm == c_norm and bool(b_norm))
        out[f"{field}_jaccard_tok"][i] = jaccard(b_tok, c_tok)
        out[f"{field}_containment_tok"][i] = _containment(b_tok, c_tok)
        out[f"{field}_jaccard_ngram"][i] = jaccard(b_ng, c_ng)
        if b_dig or c_dig:
            out[f"{field}_digits_jaccard"][i] = jaccard(b_dig, c_dig)
        else:
            out[f"{field}_digits_jaccard"][i] = -1.0  # чисел нет ни там, ни там
        out[f"{field}_lcp_ratio"][i] = _lcp_ratio(b_norm, c_norm)

        lb, lc = len(b_norm), len(c_norm)
        out[f"{field}_len_ratio"][i] = min(lb, lc) / max(lb, lc) if max(lb, lc) else 0.0
        out[f"{field}_len_diff"][i] = abs(lb - lc)

    return out


def _parse_params(raw: str) -> dict:
    if not raw:
        return {}
    try:
        parsed = json.loads(raw)
    except (ValueError, TypeError):
        return {}
    return parsed if isinstance(parsed, dict) else {}


def _json_params_block(base_raw: np.ndarray, cand_raw: np.ndarray) -> dict[str, np.ndarray]:
    n = len(base_raw)
    out = {
        "params_n_common_keys": np.zeros(n, dtype=np.float32),
        "params_key_jaccard": np.zeros(n, dtype=np.float32),
        "params_value_match_ratio": np.zeros(n, dtype=np.float32),
        "params_n_base": np.zeros(n, dtype=np.float32),
        "params_n_cand": np.zeros(n, dtype=np.float32),
    }
    cache = _BoundedCache(_parse_params)

    for i in range(n):
        b, c = cache.get(base_raw[i]), cache.get(cand_raw[i])
        out["params_n_base"][i] = len(b)
        out["params_n_cand"][i] = len(c)
        if not b or not c:
            out["params_value_match_ratio"][i] = -1.0
            continue
        bk, ck = set(b), set(c)
        common = bk & ck
        out["params_n_common_keys"][i] = len(common)
        out["params_key_jaccard"][i] = len(common) / len(bk | ck)
        if common:
            equal = sum(1 for k in common if b[k] == c[k])
            out["params_value_match_ratio"][i] = equal / len(common)
        else:
            out["params_value_match_ratio"][i] = -1.0

    return out


def _clean_price(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
    return np.where(values > 0, values, np.nan)


def side_numeric_features(df: pd.DataFrame, prefix: str) -> np.ndarray:
    price = _clean_price(df[f"{prefix}_price"])
    log_price = np.log1p(np.nan_to_num(price, nan=0.0))
    price_known = np.isfinite(price).astype(np.float64)

    images = pd.to_numeric(df[f"{prefix}_count_images"], errors="coerce").fillna(0).to_numpy()
    title_len = df[f"{prefix}_title"].fillna("").str.len().to_numpy()
    desc_len = df[f"{prefix}_description"].fillna("").str.len().to_numpy()

    out = np.column_stack([
        log_price, price_known, images, np.log1p(title_len), np.log1p(desc_len)
    ]).astype(np.float32)
    if not np.isfinite(out).all():
        raise ValueError(f"в признаках стороны {prefix} остались NaN или бесконечности")
    return out


def _numeric_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # цена, число фотографий и готовые гео-флаги
    out: dict[str, np.ndarray] = {}

    bp = _clean_price(df["base_price"])
    cp = _clean_price(df["cand_price"])
    lo, hi = np.minimum(bp, cp), np.maximum(bp, cp)
    with np.errstate(divide="ignore", invalid="ignore"):
        out["price_ratio"] = np.where(hi > 0, lo / hi, np.nan)
        out["price_log_diff"] = np.abs(np.log1p(bp) - np.log1p(cp))
    out["price_abs_diff"] = np.abs(bp - cp)
    out["price_min"] = lo
    out["price_max"] = hi
    out["price_missing"] = (np.isnan(bp) | np.isnan(cp)).astype(np.float32)

    bi = pd.to_numeric(df["base_count_images"], errors="coerce").fillna(0).to_numpy()
    ci = pd.to_numeric(df["cand_count_images"], errors="coerce").fillna(0).to_numpy()
    out["images_abs_diff"] = np.abs(bi - ci)
    out["images_min"] = np.minimum(bi, ci)
    out["images_max"] = np.maximum(bi, ci)
    out["images_equal"] = (bi == ci).astype(np.float32)

    for col in ("is_same_location", "is_same_region"):
        if col in df.columns:
            out[col] = df[col].fillna(False).astype(np.float32).to_numpy()

    return out


def _categorical_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # разные категории почти исключают дубль, то есть это работает фильтром, а не слабым сигналом
    out: dict[str, np.ndarray] = {}
    for field in ("category_name", "subcategory_name", "param1", "param2"):
        b_col, c_col = f"base_{field}", f"cand_{field}"
        if b_col not in df.columns or c_col not in df.columns:
            continue
        b, c = _safe_str(df[b_col]), _safe_str(df[c_col])
        out[f"same_{field}"] = (b == c).astype(np.float32)
        out[f"{field}_missing"] = ((b == "") | (c == "")).astype(np.float32)
    return out


def _title_image_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    out: dict[str, np.ndarray] = {}
    if "base_title_image" not in df.columns:
        return out
    b = _safe_str(df["base_title_image"])
    c = _safe_str(df["cand_title_image"])
    out["title_image_both_present"] = ((b != "") & (c != "")).astype(np.float32)
    out["title_image_equal"] = (b == c).astype(np.float32)
    return out


def _cross_block(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # перепост часто копирует заголовок в описание, значит такое вложение ловит дубли,
    # у которых сами заголовки переписаны и напрямую не совпадают
    n = len(df)
    titles = {p: _safe_str(df[f"{p}_title"]) for p in ("base", "cand")}
    descriptions = {p: _safe_str(df[f"{p}_description"]) for p in ("base", "cand")}

    title_cache = _BoundedCache(lambda raw: set(tokenize(raw)))
    desc_cache = _BoundedCache(lambda raw: set(tokenize(raw[:MAX_DESCRIPTION_CHARS])))

    out = {
        "cross_base_title_in_cand_desc": np.zeros(n, dtype=np.float32),
        "cross_cand_title_in_base_desc": np.zeros(n, dtype=np.float32),
    }
    for i in range(n):
        out["cross_base_title_in_cand_desc"][i] = _containment(
            title_cache.get(titles["base"][i]), desc_cache.get(descriptions["cand"][i])
        )
        out["cross_cand_title_in_base_desc"][i] = _containment(
            title_cache.get(titles["cand"][i]), desc_cache.get(descriptions["base"][i])
        )
    return out


def build_features(df: pd.DataFrame, *, verbose: bool = False) -> pd.DataFrame:
    blocks: dict[str, np.ndarray] = {}

    for field, ngram in TEXT_PAIRS:
        if verbose:
            print(f"  признаки по полю {field}...", flush=True)
        max_chars = MAX_DESCRIPTION_CHARS if field == "description" else None
        blocks.update(
            _text_similarity_block(
                _safe_str(df[f"base_{field}"]),
                _safe_str(df[f"cand_{field}"]),
                field,
                ngram,
                max_chars,
            )
        )

    if "base_json_params" in df.columns:
        if verbose:
            print("  признаки по json_params...", flush=True)
        blocks.update(
            _json_params_block(_safe_str(df["base_json_params"]), _safe_str(df["cand_json_params"]))
        )

    if verbose:
        print("  числовые и категориальные...", flush=True)
    blocks.update(_numeric_block(df))
    blocks.update(_categorical_block(df))
    blocks.update(_title_image_block(df))

    if verbose:
        print("  перекрёстные заголовок/описание...", flush=True)
    blocks.update(_cross_block(df))

    out = pd.DataFrame(blocks, index=df.index)
    return out.astype(np.float32)


def add_embedding_features(
    features: pd.DataFrame,
    base_emb: np.ndarray,
    cand_emb: np.ndarray,
    prefix: str = "emb",
) -> pd.DataFrame:
    b = np.ascontiguousarray(base_emb, dtype=np.float32)
    c = np.ascontiguousarray(cand_emb, dtype=np.float32)
    bn = b / np.maximum(np.linalg.norm(b, axis=1, keepdims=True), 1e-8)
    cn = c / np.maximum(np.linalg.norm(c, axis=1, keepdims=True), 1e-8)

    diff = np.abs(bn - cn)
    features = features.copy()
    features[f"{prefix}_cosine"] = (bn * cn).sum(axis=1).astype(np.float32)
    features[f"{prefix}_l2"] = np.linalg.norm(bn - cn, axis=1).astype(np.float32)
    features[f"{prefix}_diff_mean"] = diff.mean(axis=1).astype(np.float32)
    features[f"{prefix}_diff_max"] = diff.max(axis=1).astype(np.float32)
    features[f"{prefix}_diff_std"] = diff.std(axis=1).astype(np.float32)
    return features


def embedding_pair_features(
    embeddings: np.ndarray,
    base_idx: np.ndarray,
    cand_idx: np.ndarray,
    prefix: str = "emb",
    chunk: int = 200_000,
    index: pd.Index | None = None,
) -> pd.DataFrame:
    n = len(base_idx)
    cols = ["cosine", "l2", "diff_mean", "diff_max", "diff_std"]
    out = {f"{prefix}_{c}": np.empty(n, dtype=np.float32) for c in cols}

    def take(idx: np.ndarray) -> np.ndarray:
        rows = embeddings[idx].astype(np.float32, copy=False)
        norms = np.maximum(np.linalg.norm(rows, axis=1, keepdims=True), 1e-8)
        return rows / norms

    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        b = take(base_idx[start:end])
        c = take(cand_idx[start:end])
        diff = np.abs(b - c)
        out[f"{prefix}_cosine"][start:end] = (b * c).sum(axis=1)
        out[f"{prefix}_l2"][start:end] = np.linalg.norm(b - c, axis=1)
        out[f"{prefix}_diff_mean"][start:end] = diff.mean(axis=1)
        out[f"{prefix}_diff_max"][start:end] = diff.max(axis=1)
        out[f"{prefix}_diff_std"][start:end] = diff.std(axis=1)

    return pd.DataFrame(out, index=index)

In [ ]:
"""одно объявление стоит и как base, и как cand, поэтому случайное разбиение по строкам
растаскивает его между train и valid и задирает оценку. режу по group_id, кластером целиком.
"""


import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

__all__ = ["group_folds", "leakage_report"]


def group_folds(
    groups: np.ndarray, n_splits: int = 5, seed: int = 42
) -> list[tuple[np.ndarray, np.ndarray]]:
    groups = np.asarray(groups)
    # GroupKFold раскладывает группы по размеру и не перемешивает, то есть сид на него
    # не влияет — перемешиваю сам, переименовав группы случайной перестановкой
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    shuffled = rng.permutation(len(uniq))
    remap = dict(zip(uniq, shuffled))
    permuted = np.array([remap[g] for g in groups])

    splitter = GroupKFold(n_splits=n_splits)
    dummy = np.zeros(len(groups))
    return list(splitter.split(dummy, groups=permuted))


def leakage_report(
    df: pd.DataFrame,
    train_idx: np.ndarray,
    valid_idx: np.ndarray,
    id_cols: tuple[str, ...] = ("base_item_id", "cand_item_id"),
) -> dict[str, float]:
    # доля объявлений валидации, которые уже были в обучении. по group_id должна быть нулевой
    train_ids: set = set()
    valid_ids: set = set()
    for col in id_cols:
        train_ids |= set(df.iloc[train_idx][col].dropna())
        valid_ids |= set(df.iloc[valid_idx][col].dropna())
    if not valid_ids:
        return {"overlap_ratio": 0.0, "n_valid_items": 0, "n_leaked_items": 0}
    leaked = valid_ids & train_ids
    return {
        "overlap_ratio": len(leaked) / len(valid_ids),
        "n_valid_items": len(valid_ids),
        "n_leaked_items": len(leaked),
    }

In [ ]:
__all__ = ["pick_device", "reset_device_cache"]

_CACHED: str | None = None


def pick_device(verbose: bool = True) -> str:
    global _CACHED
    if _CACHED is not None:
        return _CACHED

    import torch

    if not torch.cuda.is_available():
        _CACHED = "cpu"
        return _CACHED

    try:
        probe = torch.ones(8, 8, device="cuda")
        (probe @ probe).sum().item()
        _CACHED = "cuda"
        if verbose:
            print(f"устройство: cuda ({torch.cuda.get_device_name(0)})", flush=True)
    except Exception as exc:  # noqa: BLE001
        name = "неизвестна"
        try:
            name = torch.cuda.get_device_name(0)
        except Exception:  # noqa: BLE001
            pass
        if verbose:
            print(f"видеокарта {name} непригодна ({exc}); считаю на CPU", flush=True)
        _CACHED = "cpu"

    return _CACHED


def reset_device_cache() -> None:
    global _CACHED
    _CACHED = None

## embeddings

In [ ]:
"""эмбеддинги rubert-tiny2."""


import numpy as np


__all__ = ["TextIndex", "encode_texts", "DEFAULT_MODEL"]

DEFAULT_MODEL = "cointegrated/rubert-tiny2"


class TextIndex:
    def __init__(self) -> None:
        self._ids: dict[str, int] = {}
        self.texts: list[str] = []

    def add(self, text: str) -> int:
        idx = self._ids.get(text)
        if idx is None:
            idx = len(self.texts)
            self._ids[text] = idx
            self.texts.append(text)
        return idx

    def add_many(self, texts) -> np.ndarray:
        return np.fromiter((self.add(t) for t in texts), dtype=np.int64, count=len(texts))

    def __len__(self) -> int:
        return len(self.texts)


def build_bert_text(title: str, description: str, desc_chars: int = 128) -> str:
    title = restore_homoglyphs(title or "")
    desc = restore_homoglyphs((description or "")[:desc_chars])
    return f"{title}. {desc}".strip()


def encode_texts(
    texts: list[str],
    model_name: str = DEFAULT_MODEL,
    batch_size: int = 512,
    max_length: int = 64,
    device: str | None = None,
    verbose: bool = True,
) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer

    if device is None:
        device = pick_device(verbose=verbose)
    if verbose:
        print(f"кодирую {len(texts)} уникальных текстов на {device}", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    # float16: уникальных текстов миллионы, в float32 таблица занимает больше, чем все
    # признаки вместе. точность не нужна — дальше из векторов только косинус и статистики
    dim = model.config.hidden_size
    out = np.empty((len(texts), dim), dtype=np.float16)

    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            out[start : start + len(batch)] = pooled.float().cpu().numpy().astype(np.float16)
            if verbose and start and start % (batch_size * 200) == 0:
                print(f"  {start}/{len(texts)}", flush=True)

    return out

## image comparison

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np


__all__ = [
    "ImageLocator",
    "ImageIndex",
    "DEFAULT_CLIP_MODEL",
    "dhash",
    "dhash_many",
    "hamming",
    "encode_images",
    "image_pair_features",
]

DEFAULT_CLIP_MODEL = "openai/clip-vit-base-patch32"

IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".webp")

DHASH_SIZE = 8

DRAFT_SIDE = 256


def _scandir_names(directory: Path, suffixes: tuple[str, ...]) -> list[tuple[str, str]]:
    out: list[tuple[str, str]] = []
    try:
        with os.scandir(directory) as it:
            for entry in it:
                stem, suffix = os.path.splitext(entry.name)
                if suffix.lower() in suffixes:
                    out.append((stem, suffix))
    except OSError:
        return []
    return out


class ImageLocator:
    def __init__(self, roots, *, suffixes=IMAGE_SUFFIXES, verbose: bool = False) -> None:
        self._roots = [Path(r) for r in roots if Path(r).exists()]
        self._suffixes = tuple(s.lower() for s in suffixes)
        self._verbose = verbose
        self._chunks: list[Path] = []
        self._suffix_of_chunk: list[str] = []
        self._position: np.ndarray | None = None
        self._keys: list[str] = []

    @classmethod
    def from_roots(cls, *roots, verbose: bool = False) -> "ImageLocator":
        return cls(roots, verbose=verbose)

    def _chunk_dirs(self) -> list[Path]:
        found: list[Path] = []
        for root in self._roots:
            found.append(root)
            try:
                with os.scandir(root) as it:
                    for entry in it:
                        if entry.is_dir():
                            found.append(Path(entry.path))
            except OSError:
                continue
        return found

    def build(self, keys, *, time_budget_s: float | None = None) -> "ImageLocator":
        import time

        self._keys = list(keys)
        wanted = {k: i for i, k in enumerate(self._keys) if k}
        self._position = np.full(len(self._keys), -1, dtype=np.int32)
        if not wanted:
            return self

        started = time.time()
        remaining = len(wanted)
        for chunk in self._chunk_dirs():
            if not remaining:
                break
            if time_budget_s is not None and time.time() - started > time_budget_s:
                if self._verbose:
                    print(
                        f"индекс фотографий: бюджет исчерпан, найдено "
                        f"{len(wanted) - remaining} из {len(wanted)}",
                        flush=True,
                    )
                break
            names = _scandir_names(chunk, self._suffixes)
            if not names:
                continue
            chunk_id = len(self._chunks)
            hits = 0
            suffix_seen = names[0][1]
            for stem, suffix in names:
                position = wanted.pop(stem, None)
                if position is None:
                    continue
                self._position[position] = chunk_id
                suffix_seen = suffix
                hits += 1
            remaining -= hits
            self._chunks.append(chunk)
            self._suffix_of_chunk.append(suffix_seen)
            if self._verbose and hits:
                print(
                    f"  {chunk.name}: {len(names)} файлов, из них нужных {hits}; "
                    f"осталось найти {remaining}",
                    flush=True,
                )

        if self._verbose:
            found = int((self._position >= 0).sum())
            print(
                f"индекс фотографий готов за {time.time() - started:.0f}s: "
                f"{found} из {len(wanted) + found} ключей найдено "
                f"в {len(self._chunks)} каталогах",
                flush=True,
            )
        return self

    def path_at(self, position: int) -> Path | None:
        if self._position is None or position >= len(self._position):
            return None
        chunk_id = int(self._position[position])
        if chunk_id < 0:
            return None
        return self._chunks[chunk_id] / (self._keys[position] + self._suffix_of_chunk[chunk_id])

    def get(self, key: str) -> Path | None:
        if not key or self._position is None:
            return None
        try:
            position = self._keys.index(key)
        except ValueError:
            return None
        return self.path_at(position)

    @property
    def found_count(self) -> int:
        return 0 if self._position is None else int((self._position >= 0).sum())

    def stats(self) -> dict[str, int]:
        return {"chunks": len(self._chunks), "found": self.found_count}

    def __contains__(self, key: object) -> bool:
        return isinstance(key, str) and self.get(key) is not None

    def __len__(self) -> int:
        return self.found_count


class ImageIndex:
    def __init__(self) -> None:
        self._ids: dict[str, int] = {}
        self.keys: list[str] = []

    def add(self, key: str) -> int:
        if not key:
            return -1
        idx = self._ids.get(key)
        if idx is None:
            idx = len(self.keys)
            self._ids[key] = idx
            self.keys.append(key)
        return idx

    def add_many(self, keys) -> np.ndarray:
        return np.fromiter((self.add(k) for k in keys), dtype=np.int64, count=len(keys))

    def __len__(self) -> int:
        return len(self.keys)


def dhash(image, size: int = DHASH_SIZE) -> np.uint64:
    from PIL import Image

    if size * size > 64:
        raise ValueError(f"хеш не влезает в 64 бита: size={size} даёт {size * size} бит")

    if not isinstance(image, Image.Image):
        image = Image.open(image)
    gray = image.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS)
    pixels = np.asarray(gray, dtype=np.int16)
    bits = pixels[:, 1:] > pixels[:, :-1]
    packed = np.packbits(bits.flatten(), bitorder="big")
    return np.uint64(int.from_bytes(packed.tobytes().rjust(8, b"\x00"), "big"))


def dhash_many(
    keys: list[str],
    locator: ImageLocator,
    *,
    size: int = DHASH_SIZE,
    time_budget_s: float | None = None,
    verbose: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    import time

    hashes = np.zeros(len(keys), dtype=np.uint64)
    found = np.zeros(len(keys), dtype=bool)
    started = time.time()
    for i, key in enumerate(keys):
        if time_budget_s is not None and i % 1000 == 0 and time.time() - started > time_budget_s:
            if verbose:
                print(
                    f"бюджет исчерпан: хеши сняты для {i} из {len(keys)} фотографий",
                    flush=True,
                )
            break
        path = locator.path_at(i)
        if path is None:
            continue
        try:
            hashes[i] = dhash(path, size=size)
            found[i] = True
        except Exception:
            continue
        if verbose and i and i % 50_000 == 0:
            print(f"  хеши: {i}/{len(keys)} за {(time.time() - started) / 60:.0f} мин", flush=True)
    return hashes, found


def hamming(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    xor = np.bitwise_xor(left.astype(np.uint64), right.astype(np.uint64))
    return np.unpackbits(xor.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1).astype(np.float32)


def _as_feature_tensor(raw):
    for attribute in ("image_embeds", "pooler_output"):
        value = getattr(raw, attribute, None)
        if value is not None:
            return value
    hidden = getattr(raw, "last_hidden_state", None)
    if hidden is not None:
        return hidden[:, 0]
    return raw


def encode_images(
    keys: list[str],
    locator: ImageLocator,
    *,
    model_name: str = DEFAULT_CLIP_MODEL,
    batch_size: int = 256,
    workers: int = 16,
    device: str | None = None,
    with_hashes: bool = True,
    time_budget_s: float | None = None,
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    import time

    import torch
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor

    if device is None:
        device = pick_device(verbose=verbose)
    if verbose:
        print(f"кодирую {len(keys)} уникальных фотографий на {device}", flush=True)

    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device).eval()
    dim = model.config.projection_dim

    out = np.zeros((len(keys), dim), dtype=np.float16)
    hashes = np.zeros(len(keys), dtype=np.uint64)
    found = np.zeros(len(keys), dtype=bool)
    started = time.time()
    stopped_at = None

    def load_one(position: int):
        path = locator.path_at(position)
        if path is None:
            return None
        try:
            image = Image.open(path)
            image.draft("RGB", (DRAFT_SIDE, DRAFT_SIDE))
            image.load()
        except Exception:
            return None
        digest = np.uint64(0)
        if with_hashes:
            try:
                digest = dhash(image)
            except Exception:
                digest = np.uint64(0)
        try:
            rgb = image.convert("RGB")
        except Exception:
            return None
        finally:
            image.close()
        return position, rgb, digest

    pool = ThreadPoolExecutor(max_workers=workers)

    with torch.inference_mode():
        for start in range(0, len(keys), batch_size):
            if time_budget_s is not None and time.time() - started > time_budget_s:
                stopped_at = start
                break
            chunk_positions = range(start, min(start + batch_size, len(keys)))
            images, positions = [], []
            for loaded in pool.map(load_one, chunk_positions):
                if loaded is None:
                    continue
                position, rgb, digest = loaded
                hashes[position] = digest
                images.append(rgb)
                positions.append(position)
            if not images:
                continue
            enc = processor(images=images, return_tensors="pt").to(device)
            features = _as_feature_tensor(model.get_image_features(**enc))
            features = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-9)
            out[positions] = features.float().cpu().numpy().astype(np.float16)
            found[positions] = True
            for image in images:
                image.close()
            if verbose and start and start % (batch_size * 100) == 0:
                elapsed = time.time() - started
                print(f"  {start}/{len(keys)} за {elapsed / 60:.0f} мин", flush=True)

    pool.shutdown(wait=False)

    if verbose:
        elapsed = time.time() - started
        if stopped_at is not None:
            share = stopped_at / max(len(keys), 1)
            print(
                f"бюджет {time_budget_s / 60:.0f} мин исчерпан: закодировано "
                f"{stopped_at} из {len(keys)} ({share:.0%}), остальные пары "
                f"пойдут без визуальных признаков",
                flush=True,
            )
        else:
            print(
                f"закодировано {int(found.sum())} из {len(keys)} за {elapsed / 60:.0f} мин",
                flush=True,
            )

    return out, hashes, found


def image_pair_features(
    base_idx: np.ndarray,
    cand_idx: np.ndarray,
    *,
    hashes: np.ndarray | None = None,
    hash_found: np.ndarray | None = None,
    embeddings: np.ndarray | None = None,
    embedding_found: np.ndarray | None = None,
) -> dict[str, np.ndarray]:
    n = len(base_idx)
    out: dict[str, np.ndarray] = {}
    both_indexed = (base_idx >= 0) & (cand_idx >= 0)

    if hashes is not None:
        found = hash_found if hash_found is not None else np.ones(len(hashes), dtype=bool)
        pair_ok = both_indexed & found[base_idx] & found[cand_idx]
        distance = np.zeros(n, dtype=np.float32)
        if pair_ok.any():
            distance[pair_ok] = hamming(hashes[base_idx[pair_ok]], hashes[cand_idx[pair_ok]])
        bits = float(np.dtype(hashes.dtype).itemsize * 8)
        out["image_dhash_distance"] = np.where(pair_ok, distance / bits, 0.0).astype(np.float32)
        out["image_dhash_equal"] = (pair_ok & (distance == 0)).astype(np.float32)
        out["image_hash_pair_known"] = pair_ok.astype(np.float32)

    if embeddings is not None:
        found = (
            embedding_found
            if embedding_found is not None
            else np.ones(len(embeddings), dtype=bool)
        )
        pair_ok = both_indexed & found[base_idx] & found[cand_idx]
        cosine = np.zeros(n, dtype=np.float32)
        if pair_ok.any():
            left = embeddings[base_idx[pair_ok]].astype(np.float32)
            right = embeddings[cand_idx[pair_ok]].astype(np.float32)
            cosine[pair_ok] = np.clip((left * right).sum(axis=1), -1.0, 1.0)
        out["image_cosine"] = np.where(pair_ok, cosine, 0.0).astype(np.float32)
        out["image_embedding_pair_known"] = pair_ok.astype(np.float32)

    if not out:
        return {}

    out["image_missing"] = (~both_indexed).astype(np.float32)
    return out

## two-tower contrastive model

In [ ]:
"""косинус сырых эмбеддингов бустингу почти ничего не даёт: модель не дообучается и меряет
смысловую близость, а два зимних пуховика близки по смыслу, но не дубли. мне нужна близость
в смысле дубля, значит её надо выучивать.

замороженный энкодер даёт представление, поверх учу проекцию — она стягивает дубли и разводит
недубли. обе стороны идут через один энкодер независимо и встречаются только на расстоянии
между проекциями, то есть модель не может подсмотреть в кандидата, кодируя базу.
"""


import numpy as np


__all__ = ["TwoTowerContrastive", "contrastive_loss", "build_side_matrix", "SideView"]


class SideView:
    def __init__(self, embeddings: np.ndarray, idx: np.ndarray, numeric: np.ndarray) -> None:
        if len(idx) != len(numeric):
            raise ValueError("длина индексов и числовых признаков не совпадает")
        self.embeddings = embeddings
        self.idx = np.asarray(idx)
        self.numeric = np.asarray(numeric, dtype=np.float32)
        self.shape = (len(self.idx), embeddings.shape[1] + self.numeric.shape[1])

    def __len__(self) -> int:
        return self.shape[0]

    def take(self, rows: np.ndarray) -> np.ndarray:
        emb = self.embeddings[self.idx[rows]].astype(np.float32, copy=False)
        return np.hstack([emb, self.numeric[rows]])

    def subset(self, rows: np.ndarray) -> "SideView":
        # вид на подмножество — нарезка по фолдам без копирования
        return SideView(self.embeddings, self.idx[rows], self.numeric[rows])


class _DenseView:
    # тот же интерфейс поверх обычного массива, чтобы модель не различала случаи
    def __init__(self, array: np.ndarray) -> None:
        self.array = np.asarray(array, dtype=np.float32)
        self.shape = self.array.shape

    def __len__(self) -> int:
        return len(self.array)

    def take(self, rows: np.ndarray) -> np.ndarray:
        return self.array[rows]

    def subset(self, rows: np.ndarray) -> "_DenseView":
        return _DenseView(self.array[rows])


def _as_view(x):
    return x if hasattr(x, "take") and hasattr(x, "subset") else _DenseView(x)


def _torch():
    import torch

    return torch


def contrastive_loss(distance, label, margin: float = 1.0):
    # дубли штрафуются за расстояние, недубли — за то, что подошли ближе зазора. недубли
    # дальше margin вклада не дают, иначе обучение уходит в раздувание расстояний
    torch = _torch()
    positive = label * distance.pow(2)
    negative = (1.0 - label) * torch.clamp(margin - distance, min=0.0).pow(2)
    return (positive + negative).mean()


def _build_module(input_dim: int, hidden: int, output: int, dropout: float):
    torch = _torch()
    nn = torch.nn

    class TwoTower(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.BatchNorm1d(hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, output),
            )

        def encode(self, x):
            z = self.encoder(x)
            # l2: без неё расстояние можно уменьшать, просто сжимая представление,
            # и зазор перестаёт что-либо значить
            return z / z.norm(dim=1, keepdim=True).clamp(min=1e-8)

        def forward(self, x_base, x_cand):
            z_base = self.encode(x_base)
            z_cand = self.encode(x_cand)
            delta = z_base - z_cand
            return torch.sqrt((delta * delta).sum(dim=1) + 1e-12)

    return TwoTower()


class TwoTowerContrastive:
    def __init__(
        self,
        input_dim: int,
        hidden: int = 256,
        output: int = 128,
        margin: float = 1.0,
        dropout: float = 0.1,
        lr: float = 1e-3,
        batch_size: int = 4096,
        epochs: int = 5,
        seed: int = 42,
        device: str | None = None,
        verbose: bool = True,
    ) -> None:
        self.input_dim = input_dim
        self.hidden = hidden
        self.output = output
        self.margin = margin
        self.dropout = dropout
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.seed = seed
        self.verbose = verbose
        self.device = device or pick_device(verbose=verbose)
        self.model = None

    def fit(self, x_base, x_cand, y: np.ndarray) -> "TwoTowerContrastive":
        torch = _torch()
        torch.manual_seed(self.seed)

        base_view, cand_view = _as_view(x_base), _as_view(x_cand)
        self.model = _build_module(self.input_dim, self.hidden, self.output, self.dropout)
        self.model = self.model.to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        y = np.asarray(y, dtype=np.float32)
        n = len(y)
        # дублей ~5%: без выравнивания батч почти целиком из недублей, градиент от
        # положительной части тонет в шуме и модель сходится к «всё далеко»
        positive_idx = np.flatnonzero(y == 1)
        negative_idx = np.flatnonzero(y == 0)
        if len(positive_idx) == 0 or len(negative_idx) == 0:
            raise ValueError("в обучающей выборке должны быть оба класса")
        rng = np.random.default_rng(self.seed)
        half = max(self.batch_size // 2, 1)
        steps = max(n // self.batch_size, 1)

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total = 0.0
            for _ in range(steps):
                rows = np.concatenate([
                    rng.choice(positive_idx, size=half, replace=len(positive_idx) < half),
                    rng.choice(negative_idx, size=half, replace=len(negative_idx) < half),
                ])
                xb = torch.from_numpy(base_view.take(rows)).to(self.device)
                xc = torch.from_numpy(cand_view.take(rows)).to(self.device)
                yt = torch.from_numpy(y[rows]).to(self.device)

                optimizer.zero_grad()
                loss = contrastive_loss(self.model(xb, xc), yt, self.margin)
                loss.backward()
                optimizer.step()
                total += loss.item()
            if self.verbose:
                print(f"  эпоха {epoch}: loss {total / steps:.4f}", flush=True)
        return self

    def predict(self, x_base, x_cand, batch_size: int = 16384) -> np.ndarray:
        # 1/(1+d), а не -d: так значения лежат в (0, 1] и годятся как признак наравне с остальными
        torch = _torch()
        if self.model is None:
            raise RuntimeError("модель не обучена")
        base_view, cand_view = _as_view(x_base), _as_view(x_cand)
        self.model.eval()
        n = len(base_view)
        out = np.empty(n, dtype=np.float32)
        with torch.inference_mode():
            for start in range(0, n, batch_size):
                rows = np.arange(start, min(start + batch_size, n))
                b = torch.from_numpy(base_view.take(rows)).to(self.device)
                c = torch.from_numpy(cand_view.take(rows)).to(self.device)
                distance = self.model(b, c).cpu().numpy()
                out[rows] = 1.0 / (1.0 + distance)
        return out


def build_side_matrix(
    embeddings: np.ndarray, idx: np.ndarray, numeric: np.ndarray
) -> np.ndarray:
    # вход одной башни: эмбеддинг плюс числовые атрибуты. стандартизует вызывающий код
    emb = embeddings[idx].astype(np.float32, copy=False)
    return np.hstack([emb, numeric.astype(np.float32, copy=False)])

## cross-encoder

In [ ]:
"""башни кодируют стороны независимо и встречаются только на расстоянии, значит сопоставить
слово базы со словом кандидата они не могут. для дублей это дорого: iphone 11 256 гб и
iphone 11 128 гб дают почти одинаковые векторы, а различает их одно число.

кросс-энкодер получает обе стороны одной последовательностью через [SEP], внимание работает
между ними. цена — представление больше не посчитать заранее на объявление, скор считается
на каждую пару. тут дообучается сам энкодер, а не проекция над ним.
"""


import time

import numpy as np


__all__ = ["CrossEncoderRanker", "DEFAULT_MODEL"]

DEFAULT_MODEL = "cointegrated/rubert-tiny2"


class CrossEncoderRanker:
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        max_length: int = 128,
        batch_size: int = 128,
        epochs: int = 2,
        lr: float = 5e-5,
        negative_ratio: float | None = 4.0,
        max_seconds: float | None = None,
        seed: int = 42,
        device: str | None = None,
        verbose: bool = True,
    ) -> None:
        self.model_name = model_name
        self.max_length = max_length
        self.batch_size = batch_size
        self.epochs = epochs
        self.lr = lr
        self.negative_ratio = negative_ratio
        self.max_seconds = max_seconds
        self.seed = seed
        self.verbose = verbose
        self.device = device or pick_device(verbose=verbose)
        self.model = None
        self.tokenizer = None
        self.history: dict = {}

    def _subsample(self, y: np.ndarray) -> np.ndarray:
        # все позитивы и часть негативов
        positives = np.flatnonzero(y == 1)
        negatives = np.flatnonzero(y == 0)
        if self.negative_ratio is None:
            return np.concatenate([positives, negatives])
        keep = min(len(negatives), int(len(positives) * self.negative_ratio))
        rng = np.random.default_rng(self.seed)
        chosen = rng.choice(negatives, size=keep, replace=False)
        return np.concatenate([positives, chosen])

    def fit(
        self, texts: list[str], idx_a: np.ndarray, idx_b: np.ndarray, y: np.ndarray
    ) -> "CrossEncoderRanker":
        import torch
        from torch.nn import BCEWithLogitsLoss
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        if self.device != "cuda":
            raise RuntimeError(
                "кросс-энкодер требует рабочей видеокарты: на процессоре дообучение "
                "на миллионах пар не укладывается ни в какой разумный бюджет"
            )

        torch.manual_seed(self.seed)
        y = np.asarray(y, dtype=np.float32)
        index = self._subsample(y)
        rng = np.random.default_rng(self.seed)
        rng.shuffle(index)

        # после прореживания доля позитивов выросла, значит вес класса считаю по тому,
        # что реально попало в обучение, а не по исходной выборке
        n_pos = float((y[index] == 1).sum())
        n_neg = float((y[index] == 0).sum())
        pos_weight = torch.tensor([n_neg / max(n_pos, 1.0)], device=self.device)

        if self.verbose:
            print(
                f"  кросс-энкодер: {len(index)} пар из {len(y)} "
                f"(позитивов {n_pos:.0f}, негативов {n_neg:.0f}, вес {pos_weight.item():.2f})",
                flush=True,
            )

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=1
        ).to(self.device)

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        loss_fn = BCEWithLogitsLoss(pos_weight=pos_weight)
        scaler = torch.amp.GradScaler("cuda")

        steps_per_epoch = max(len(index) // self.batch_size, 1)
        total_steps = steps_per_epoch * self.epochs
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.lr, total_steps=total_steps, pct_start=0.1
        )

        started = time.time()
        stopped_early = False
        done_steps = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            rng.shuffle(index)
            running = 0.0
            for step in range(steps_per_epoch):
                rows = index[step * self.batch_size : (step + 1) * self.batch_size]
                batch = self.tokenizer(
                    [texts[i] for i in idx_a[rows]],
                    [texts[i] for i in idx_b[rows]],
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt",
                ).to(self.device)
                target = torch.from_numpy(y[rows]).to(self.device).unsqueeze(1)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = self.model(**batch).logits
                    loss = loss_fn(logits.float(), target)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                running += loss.item()
                done_steps += 1

                if self.max_seconds and time.time() - started > self.max_seconds:
                    stopped_early = True
                    break

            if self.verbose:
                print(
                    f"    эпоха {epoch}: loss {running / max(step + 1, 1):.4f}, "
                    f"{time.time() - started:.0f}s",
                    flush=True,
                )
            if stopped_early:
                break

        self.history = {
            "seconds": time.time() - started,
            "steps_done": done_steps,
            "steps_planned": total_steps,
            "stopped_early": stopped_early,
            "train_pairs": int(len(index)),
        }
        if stopped_early and self.verbose:
            print(
                f"  БЮДЖЕТ ИСЧЕРПАН: пройдено {done_steps} шагов из {total_steps} "
                f"({done_steps / total_steps:.0%})",
                flush=True,
            )
        return self

    def predict(
        self, texts: list[str], idx_a: np.ndarray, idx_b: np.ndarray, batch_size: int = 512
    ) -> np.ndarray:
        import torch

        if self.model is None:
            raise RuntimeError("модель не обучена")
        self.model.eval()
        out = np.empty(len(idx_a), dtype=np.float32)

        with torch.inference_mode():
            for start in range(0, len(idx_a), batch_size):
                end = min(start + batch_size, len(idx_a))
                batch = self.tokenizer(
                    [texts[i] for i in idx_a[start:end]],
                    [texts[i] for i in idx_b[start:end]],
                    padding=True,
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors="pt",
                ).to(self.device)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = self.model(**batch).logits
                out[start:end] = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()
        return out

## blending

In [ ]:
import numpy as np

__all__ = ["rank_normalize", "rank_blend"]


def rank_normalize(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 0:
        return np.array([], dtype=np.float64)
    if n == 1:
        return np.array([0.5])

    order = np.argsort(scores, kind="stable")
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(n, dtype=np.float64)

    # одинаковым значениям — средний ранг, иначе порядок внутри ties зависел бы от порядка строк
    sorted_scores = scores[order]
    starts = np.flatnonzero(np.concatenate(([True], sorted_scores[1:] != sorted_scores[:-1])))
    sizes = np.diff(np.concatenate((starts, [n])))
    group_mean = np.repeat(
        (starts + (starts + sizes - 1)) / 2.0, sizes
    )
    ranks[order] = group_mean

    return ranks / (n - 1)


def rank_blend(scores_a: np.ndarray, scores_b: np.ndarray, weight: float = 0.5) -> np.ndarray:
    # weight — вес первого набора: 0.0 только второй, 1.0 только первый
    if not 0.0 <= weight <= 1.0:
        raise ValueError("вес должен лежать в [0, 1]")
    return weight * rank_normalize(scores_a) + (1.0 - weight) * rank_normalize(scores_b)

## config

`QUICK_RUN` гоняет pipeline на одной части обучающих данных

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

MAX_ENTRIES_PER_DIR = 256
MAX_DIRS_VISITED = 400


def scan_dir(directory: Path, limit: int = MAX_ENTRIES_PER_DIR):
    files: list[Path] = []
    dirs: list[Path] = []
    try:
        with os.scandir(directory) as it:
            for i, entry in enumerate(it):
                if i >= limit:
                    break
                (dirs if entry.is_dir() else files).append(Path(entry.path))
    except OSError:
        pass
    return files, dirs


def walk_shallow(root: Path, max_depth: int = 4, skip=lambda p: False):
    queue: list[tuple[Path, int]] = [(root, 0)]
    visited = 0
    while queue and visited < MAX_DIRS_VISITED:
        directory, depth = queue.pop(0)
        if skip(directory):
            continue
        visited += 1
        files, dirs = scan_dir(directory)
        yield directory, files, dirs
        if depth < max_depth:
            queue.extend((d, depth + 1) for d in dirs)


def find_data_dir(root: Path = INPUT_ROOT) -> Path:
    seen: list[Path] = []
    for directory, files, _dirs in walk_shallow(
        root, skip=lambda p: "image" in p.name.lower()
    ):
        seen.append(directory)
        if any(f.name.startswith("train_part_") and f.suffix == ".parquet" for f in files):
            return directory
    listing = "\n".join(f"  {p}" for p in seen[:40])
    raise FileNotFoundError(
        f"под {root} нет файлов train_part_*.parquet.\n"
        f"Подключите датасет chuvirla/avito-ml-cup-default-dataset.\n"
        f"Просмотрены каталоги:\n{listing or '  (пусто)'}"
    )


def find_image_dirs(root: Path = INPUT_ROOT) -> list[Path]:
    found: list[Path] = []
    for directory, _files, dirs in walk_shallow(
        root, max_depth=3, skip=lambda p: any(p == f or f in p.parents for f in found)
    ):
        for candidate in dirs:
            if "image" in candidate.name.lower() and candidate not in found:
                found.append(candidate)
    return sorted(found)


DATA_DIR = find_data_dir()
print("данные:", DATA_DIR)

QUICK_RUN = False
USE_BERT = True
USE_TWO_TOWER = True
USE_CROSS_ENCODER = True
USE_IMAGES = True
USE_IMAGE_CLIP = True
N_SPLITS = 5
SEED = 42

CROSS_ENCODER_SECONDS_PER_FOLD = 900
CROSS_ENCODER_EPOCHS = 2
CROSS_ENCODER_NEGATIVE_RATIO = 4.0

TRAIN_FILES = sorted(DATA_DIR.glob("train_part_*.parquet"))
TEST_FILES = sorted(DATA_DIR.glob("test_part_*.parquet"))
if QUICK_RUN:
    TRAIN_FILES = TRAIN_FILES[-1:]
    TEST_FILES = []

print("train:", [f.name for f in TRAIN_FILES])
print("test :", [f.name for f in TEST_FILES])
if not TRAIN_FILES:
    raise FileNotFoundError(f"в {DATA_DIR} нет обучающих частей")

## load and featurize

In [ ]:
FEATURE_COLUMNS_TEXT = [
    "base_title", "cand_title", "base_description", "cand_description",
    "base_category_name", "cand_category_name",
    "base_subcategory_name", "cand_subcategory_name",
    "base_param1", "cand_param1", "base_param2", "cand_param2",
    "base_json_params", "cand_json_params",
    "base_price", "cand_price", "base_count_images", "cand_count_images",
    "base_title_image", "cand_title_image",
    "is_same_location", "is_same_region",
    "base_item_id", "cand_item_id",
]


def load_and_featurize(files, text_index, image_index=None, with_labels=True):
    feats, labels, groups, base_ids, cand_ids, bidx, cidx = [], [], [], [], [], [], []
    bimg, cimg = [], []

    for path in files:
        t0 = time.time()
        available = set(pq.ParquetFile(path).schema_arrow.names)
        cols = [c for c in FEATURE_COLUMNS_TEXT if c in available]
        if with_labels:
            cols += [c for c in ("is_double", "group_id") if c in available]

        df = pd.read_parquet(path, columns=cols)
        print(f"{path.name}: {len(df)} пар", flush=True)

        feats.append(build_features(df, verbose=True))
        base_ids.append(df["base_item_id"].to_numpy())
        cand_ids.append(df["cand_item_id"].to_numpy())
        if with_labels:
            labels.append(df["is_double"].to_numpy())
            groups.append(df["group_id"].to_numpy())

        if USE_BERT:
            b_texts = [
                build_bert_text(t, d)
                for t, d in zip(df["base_title"].fillna(""), df["base_description"].fillna(""))
            ]
            c_texts = [
                build_bert_text(t, d)
                for t, d in zip(df["cand_title"].fillna(""), df["cand_description"].fillna(""))
            ]
            bidx.append(text_index.add_many(b_texts))
            cidx.append(text_index.add_many(c_texts))
            del b_texts, c_texts

        if image_index is not None:
            bimg.append(image_index.add_many(df["base_title_image"].fillna("").astype(str)))
            cimg.append(image_index.add_many(df["cand_title_image"].fillna("").astype(str)))

        del df
        gc.collect()
        print(f"  готово за {time.time() - t0:.0f}s", flush=True)

    out = {
        "X": pd.concat(feats, ignore_index=True),
        "base_id": np.concatenate(base_ids),
        "cand_id": np.concatenate(cand_ids),
    }
    if with_labels:
        out["y"] = np.concatenate(labels)
        out["group"] = np.concatenate(groups)
    if USE_BERT:
        out["bidx"] = np.concatenate(bidx)
        out["cidx"] = np.concatenate(cidx)
    if bimg:
        out["bimg"] = np.concatenate(bimg)
        out["cimg"] = np.concatenate(cimg)
    return out


text_index = TextIndex()
image_index = ImageIndex() if USE_IMAGES else None
t0 = time.time()
train = load_and_featurize(TRAIN_FILES, text_index, image_index, with_labels=True)
test = (
    load_and_featurize(TEST_FILES, text_index, image_index, with_labels=False)
    if TEST_FILES
    else None
)
print(f"\nвсего признаков собрано за {time.time() - t0:.0f}s")
print("train:", train["X"].shape, "| доля дублей: %.4f" % train["y"].mean())
if test is not None:
    print("test :", test["X"].shape)
print("уникальных текстов для трансформера:", len(text_index))

## embedding titles

считаю один раз на уникальный текст, потом разворачиваю в парные признаки кусками по
200 тысяч строк. 312 сырых координат в бустинг не подаю: информацию о паре несут
свёртки, а дерево на координатах и переобучается, и считается втрое дольше

In [ ]:
if USE_BERT:
    t0 = time.time()
    embeddings = encode_texts(text_index.texts, batch_size=512, max_length=64)
    print(f"эмбеддинги {embeddings.shape} за {time.time() - t0:.0f}s")

    train["X"] = pd.concat(
        [train["X"], embedding_pair_features(embeddings, train["bidx"], train["cidx"])],
        axis=1,
    )
    if test is not None:
        test["X"] = pd.concat(
            [test["X"], embedding_pair_features(embeddings, test["bidx"], test["cidx"])],
            axis=1,
        )
    print("признаков после эмбеддингов:", train["X"].shape[1])

## фотографии объявлений

In [ ]:
IMAGE_INDEX_SECONDS = 1800
IMAGE_CLIP_SECONDS = 14400

image_features_used = False
if USE_IMAGES and "bimg" in train:
    image_dirs = find_image_dirs()
    if not image_dirs:
        print("наборы с фотографиями не подключены — визуальные признаки пропускаю")
    else:
        print("наборы с фотографиями:", [str(p) for p in image_dirs])
        t0 = time.time()
        keys = image_index.keys
        print(f"уникальных фотографий в парах: {len(keys)}", flush=True)

        locator = ImageLocator.from_roots(*image_dirs, verbose=True).build(
            keys, time_budget_s=IMAGE_INDEX_SECONDS
        )
        if locator.found_count == 0:
            print("имена файлов не совпадают с title_image — примеры того, что лежит рядом:")
            for root in image_dirs:
                shown = []
                with os.scandir(root) as it:
                    for i, entry in enumerate(it):
                        if i >= 5:
                            break
                        shown.append(entry.name)
                print(f"  {root}: {shown}")
            print("примеры ключей из данных:", keys[:3])
            print("визуальные признаки пропускаю", flush=True)
            image_dirs = []

    if image_dirs:
        if USE_IMAGE_CLIP and pick_device(verbose=False) == "cuda":
            image_embeddings, image_hashes, image_found = encode_images(
                keys, locator, time_budget_s=IMAGE_CLIP_SECONDS
            )
        else:
            print("CLIP пропущен, считаю только перцептивные хеши", flush=True)
            image_embeddings = None
            image_hashes, image_found = dhash_many(
                keys, locator, time_budget_s=IMAGE_CLIP_SECONDS, verbose=True
            )

        for part in (train, test):
            if part is None:
                continue
            block = image_pair_features(
                part["bimg"],
                part["cimg"],
                hashes=image_hashes,
                hash_found=image_found,
                embeddings=image_embeddings,
                embedding_found=image_found if image_embeddings is not None else None,
            )
            part["X"] = pd.concat([part["X"], pd.DataFrame(block, index=part["X"].index)], axis=1)

        image_features_used = True
        print(f"признаков после фотографий: {train['X'].shape[1]}")
        print(f"фотографии обработаны за {time.time() - t0:.0f}s | локатор: {locator.stats()}",
              flush=True)

        known = train["X"]["image_hash_pair_known"].to_numpy() > 0
        print(f"пар, где нашлись обе фотографии: {known.mean():.1%}")
        for name in ("image_dhash_distance", "image_cosine"):
            if name not in train["X"].columns:
                continue
            values = train["X"][name].to_numpy()
            positive = known & (train["y"] == 1)
            negative = known & (train["y"] == 0)
            if positive.any() and negative.any():
                print(
                    f"  {name}: дубли {values[positive].mean():.3f} | "
                    f"не дубли {values[negative].mean():.3f}"
                )
        del locator
        gc.collect()

## side features for towers

башне нужен вход про само объявление, а не про пару: эмбеддинг плюс несколько чисел.
стандартизацию считаю по обеим сторонам сразу, иначе у базы и кандидата будут разные
шкалы при общем энкодере

In [ ]:
def side_numeric(files, prefix):
    parts = []
    for path in files:
        df = pd.read_parquet(
            path,
            columns=[f"{prefix}_price", f"{prefix}_count_images",
                     f"{prefix}_title", f"{prefix}_description"],
        )
        parts.append(side_numeric_features(df, prefix))
        del df
    return np.vstack(parts)


if USE_BERT and USE_TWO_TOWER:
    num_base = side_numeric(TRAIN_FILES, "base")
    num_cand = side_numeric(TRAIN_FILES, "cand")
    # статистики только по обучающим: тест в них попадать не должен, это утечка
    stats_source = np.vstack([num_base, num_cand])
    mean, std = stats_source.mean(0), stats_source.std(0) + 1e-6
    del stats_source
    gc.collect()

    def standardise(m):
        return (m - mean) / std

    tower_base = SideView(embeddings, train["bidx"], standardise(num_base))
    tower_cand = SideView(embeddings, train["cidx"], standardise(num_cand))
    print("вход башни:", tower_base.shape)

    if test is not None:
        tower_base_test = SideView(
            embeddings, test["bidx"], standardise(side_numeric(TEST_FILES, "base"))
        )
        tower_cand_test = SideView(
            embeddings, test["cidx"], standardise(side_numeric(TEST_FILES, "cand"))
        )

In [ ]:
from sklearn.model_selection import StratifiedKFold

pairs_df = pd.DataFrame({"base_item_id": train["base_id"], "cand_item_id": train["cand_id"]})

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
random_tr, random_va = next(iter(skf.split(train["X"], train["y"])))
folds = group_folds(train["group"], n_splits=N_SPLITS, seed=SEED)

print("случайное разбиение по строкам:", leakage_report(pairs_df, random_tr, random_va))
print("разбиение по group_id        :", leakage_report(pairs_df, *folds[0]))

In [ ]:
TOWER_KWARGS = dict(hidden=256, output=128, margin=1.0, epochs=8, batch_size=4096, seed=SEED)
INNER_SPLITS = 4


def tower_scores_for_fold(tr, va):
    sim_tr = np.zeros(len(tr), dtype=np.float32)
    for inner_tr, inner_va in group_folds(train["group"][tr], INNER_SPLITS, seed=SEED + 1):
        tower = TwoTowerContrastive(
            input_dim=tower_base.shape[1], verbose=False, **TOWER_KWARGS
        )
        tower.fit(
            tower_base.subset(tr[inner_tr]), tower_cand.subset(tr[inner_tr]), train["y"][tr][inner_tr]
        )
        sim_tr[inner_va] = tower.predict(
            tower_base.subset(tr[inner_va]), tower_cand.subset(tr[inner_va])
        )

    full = TwoTowerContrastive(input_dim=tower_base.shape[1], verbose=False, **TOWER_KWARGS)
    full.fit(tower_base.subset(tr), tower_cand.subset(tr), train["y"][tr])
    sim_va = full.predict(tower_base.subset(va), tower_cand.subset(va))
    return sim_tr, sim_va, full

## training

In [ ]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

LGB_PARAMS = dict(
    objective="binary",
    # метрику задаю явно, иначе lightgbm считает ещё и binary_logloss, а ранняя остановка
    # следит за всеми сразу. со scale_pos_weight логлосс деградирует с первых итераций,
    # и обучение обрывалось на втором-третьем дереве: 0.35 PR-AUC вместо 0.46
    metric="average_precision",
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=100,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    n_estimators=1200,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)


def run_cv(X, y, groups, base_ids, folds, X_test=None, label="", with_towers=False):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test)) if X_test is not None else None
    models, rows = [], []

    for i, (tr, va) in enumerate(folds):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        X_te = X_test

        if with_towers:
            sim_tr, sim_va, full_tower = tower_scores_for_fold(tr, va)
            X_tr, X_va = X_tr.copy(), X_va.copy()
            X_tr["two_tower_sim"], X_va["two_tower_sim"] = sim_tr, sim_va
            if X_test is not None:
                X_te = X_test.copy()
                X_te["two_tower_sim"] = full_tower.predict(tower_base_test, tower_cand_test)

        pos_weight = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
        model = lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=pos_weight)
        model.fit(
            X_tr, y[tr],
            eval_set=[(X_va, y[va])],
            callbacks=[
                lgb.early_stopping(100, first_metric_only=True, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        p = model.predict_proba(X_va)[:, 1]
        oof[va] = p
        models.append(model)

        rows.append({
            "fold": i + 1,
            "roc_auc": roc_auc_score(y[va], p),
            "pr_auc": average_precision_score(y[va], p),
            "map": mean_average_precision(base_ids[va], y[va], p),
            "best_iter": model.best_iteration_,
        })
        print(f"  фолд {i+1}: ROC-AUC {rows[-1]['roc_auc']:.4f} | "
              f"PR-AUC {rows[-1]['pr_auc']:.4f} | MAP {rows[-1]['map']:.4f}", flush=True)

        if X_test is not None:
            test_pred += model.predict_proba(X_te)[:, 1] / len(folds)

    report = pd.DataFrame(rows)
    print(f"\n{label} OOF: ROC-AUC {roc_auc_score(y, oof):.4f} | "
          f"PR-AUC {average_precision_score(y, oof):.4f} | "
          f"MAP {mean_average_precision(base_ids, y, oof):.4f}")
    return oof, test_pred, models, report


USE_TOWERS_IN_MODEL = USE_BERT and USE_TWO_TOWER

print("честное разбиение по group_id:")
oof, test_pred, models, fold_report = run_cv(
    train["X"], train["y"], train["group"], train["base_id"], folds,
    X_test=test["X"] if test is not None else None, label="итог",
    with_towers=USE_TOWERS_IN_MODEL,
)

In [ ]:
def split_comparison(fold_list, name):
    scores = []
    for tr, va in fold_list:
        pw = (train["y"][tr] == 0).sum() / max((train["y"][tr] == 1).sum(), 1)
        model = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": 400}, scale_pos_weight=pw)
        model.fit(train["X"].iloc[tr], train["y"][tr])
        p = model.predict_proba(train["X"].iloc[va])[:, 1]
        scores.append(average_precision_score(train["y"][va], p))
    print(f"{name:32} PR-AUC {np.mean(scores):.4f}")
    return float(np.mean(scores))


random_folds = list(skf.split(train["X"], train["y"]))
split_scores = {
    "разбиение по group_id": split_comparison(folds, "разбиение по group_id"),
    "случайное по строкам": split_comparison(random_folds, "случайное по строкам"),
}
print(
    "\nзавышение оценки утечкой: "
    f"{split_scores['случайное по строкам'] / split_scores['разбиение по group_id']:.2f}x"
)

## cross-encoder by folds

In [ ]:
if USE_CROSS_ENCODER and pick_device(verbose=False) != "cuda":
    print("кросс-энкодер пропущен: пригодной видеокарты нет")
    USE_CROSS_ENCODER = False

if USE_CROSS_ENCODER:
    ce_oof = np.zeros(len(train["y"]), dtype=np.float32)
    ce_history = []
    t0 = time.time()

    for i, (tr, va) in enumerate(folds):
        print(f"кросс-энкодер, фолд {i + 1}/{len(folds)}", flush=True)
        ranker = CrossEncoderRanker(
            epochs=CROSS_ENCODER_EPOCHS,
            negative_ratio=CROSS_ENCODER_NEGATIVE_RATIO,
            max_seconds=CROSS_ENCODER_SECONDS_PER_FOLD,
            seed=SEED,
        )
        ranker.fit(text_index.texts, train["bidx"][tr], train["cidx"][tr], train["y"][tr])
        ce_oof[va] = ranker.predict(text_index.texts, train["bidx"][va], train["cidx"][va])
        ce_history.append(ranker.history)
        print(
            f"  фолд {i + 1}: PR-AUC {average_precision_score(train['y'][va], ce_oof[va]):.4f} | "
            f"ROC-AUC {roc_auc_score(train['y'][va], ce_oof[va]):.4f}",
            flush=True,
        )
        del ranker
        gc.collect()

    ce_pr = average_precision_score(train["y"], ce_oof)
    print(f"\nкросс-энкодер OOF: PR-AUC {ce_pr:.4f} | "
          f"ROC-AUC {roc_auc_score(train['y'], ce_oof):.4f} | "
          f"всего {time.time() - t0:.0f}s")
    if any(h["stopped_early"] for h in ce_history):
        done = sum(h["steps_done"] for h in ce_history)
        planned = sum(h["steps_planned"] for h in ce_history)
        print(f"ВНИМАНИЕ: бюджет исчерпан, пройдено {done}/{planned} шагов ({done / planned:.0%})")

## blending gbdt and cross-encoder

In [ ]:
if USE_CROSS_ENCODER:
    blend_curve = {}
    for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
        score = average_precision_score(train["y"], rank_blend(oof, ce_oof, weight=w))
        blend_curve[w] = float(score)
        mark = "  <- бустинг" if w == 1.0 else ("  <- кросс-энкодер" if w == 0.0 else "")
        print(f"  вес бустинга {w:.1f}: PR-AUC {score:.4f}{mark}")

    equal_blend = average_precision_score(train["y"], rank_blend(oof, ce_oof, weight=0.5))
    best_w = max(blend_curve, key=blend_curve.get)
    print(f"\nравная смесь: PR-AUC {equal_blend:.4f}")
    print(f"лучшая точка сетки (вес {best_w:.1f}): {blend_curve[best_w]:.4f} — "
          f"подобрана на этих же данных, как результат не берётся")

    np.savez_compressed(
        OUTPUT_DIR / "oof_predictions.npz",
        y=train["y"], gbdt=oof, cross_encoder=ce_oof, base_id=train["base_id"],
    )

In [ ]:
ABLATION_TR, ABLATION_VA = folds[0]

# скор башен для этого фолда считаю один раз на все конфигурации
if USE_TOWERS_IN_MODEL:
    _sim_tr, _sim_va, _ = tower_scores_for_fold(ABLATION_TR, ABLATION_VA)
    ablation_train = train["X"].iloc[ABLATION_TR].copy()
    ablation_valid = train["X"].iloc[ABLATION_VA].copy()
    ablation_train["two_tower_sim"] = _sim_tr
    ablation_valid["two_tower_sim"] = _sim_va
else:
    ablation_train = train["X"].iloc[ABLATION_TR]
    ablation_valid = train["X"].iloc[ABLATION_VA]

y_tr, y_va = train["y"][ABLATION_TR], train["y"][ABLATION_VA]
pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)


def ablation(drop_prefixes, name):
    cols = [c for c in ablation_train.columns
            if not any(c.startswith(p) for p in drop_prefixes)]
    model = lgb.LGBMClassifier(**{**LGB_PARAMS, "n_estimators": 400}, scale_pos_weight=pos_weight)
    model.fit(ablation_train[cols], y_tr)
    p = model.predict_proba(ablation_valid[cols])[:, 1]
    score = average_precision_score(y_va, p)
    print(f"{name:34} признаков {len(cols):3} | PR-AUC {score:.4f}", flush=True)
    return score


print("абляция (фолд 1, 400 деревьев):")
ablation_scores = {
    name: ablation(prefixes, name)
    for name, prefixes in [
        ("все признаки", []),
        ("без эмбеддингов", ["emb_"]),
        ("без скора башен", ["two_tower_"]),
        ("без эмбеддингов и башен", ["emb_", "two_tower_"]),
        ("без текстового сходства", ["title_", "description_", "cross_"]),
        ("без цены/фото/гео", ["price_", "images_", "is_same"]),
        ("без json_params", ["params_"]),
    ]
    + ([("без сравнения фотографий", ["image_"])] if image_features_used else [])
}

## feature importance

In [ ]:
importance = pd.DataFrame({
    "feature": models[0].booster_.feature_name(),
    "gain": np.mean([m.booster_.feature_importance("gain") for m in models], axis=0),
})
importance = importance.sort_values("gain", ascending=False).reset_index(drop=True)
print(importance.head(20).to_string(index=False))

## submit

In [ ]:
if test is not None:
    submission = pd.DataFrame({
        "base_id": test["base_id"],
        "cand_id": test["cand_id"],
        "probability": test_pred,
    })
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("submission.csv:", submission.shape)
    print(submission.head())

results = {
    "n_train_pairs": int(len(train["y"])),
    "positive_rate": float(train["y"].mean()),
    "n_features": int(train["X"].shape[1]),
    "n_unique_texts": int(len(text_index)),
    "use_bert": bool(USE_BERT),
    "group_split": {
        "roc_auc": float(roc_auc_score(train["y"], oof)),
        "pr_auc": float(average_precision_score(train["y"], oof)),
        "map": float(mean_average_precision(train["base_id"], train["y"], oof)),
        "per_fold": fold_report.to_dict("records"),
    },
    "split_comparison_pr_auc": split_scores,
    "leakage": {
        "random_split": leakage_report(pairs_df, random_tr, random_va),
        "group_split": leakage_report(pairs_df, *folds[0]),
    },
    "ablation_pr_auc": ablation_scores,
    "top_features": importance.head(20).to_dict("records"),
}
if USE_CROSS_ENCODER:
    results["cross_encoder"] = {
        "pr_auc": float(ce_pr),
        "roc_auc": float(roc_auc_score(train["y"], ce_oof)),
        "map": float(mean_average_precision(train["base_id"], train["y"], ce_oof)),
        "blend_curve_pr_auc": blend_curve,
        "equal_blend_pr_auc": float(equal_blend),
        "budget": ce_history,
    }
with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=float)
print(json.dumps(results["group_split"], ensure_ascii=False, indent=2, default=float))